In [2]:
import sys
import os
import pandas as pd
import getpass
import torch

from google.colab import userdata
openai_key = userdata.get('openai_key')

In [ ]:
!pip -q install backoff

In [7]:
!git clone https://github.com/bencejdanko/bert-tweeteval

# Add src to path
sys.path.append(os.path.abspath('/content/bert-tweeteval/src'))

from download import download_and_split_dataset
from llm_eval import LLMEvaluator, PROMPT_1_MINIMAL, PROMPT_2_STRUCTURED, LABELS

fatal: destination path 'bert-tweeteval' already exists and is not an empty directory.


In [8]:
# Load dataset
train_df, val_df, test_df = download_and_split_dataset()
print(f"Test set size: {len(test_df)}")

README.md: 0.00B [00:00, ?B/s]

emotion/train-00000-of-00001.parquet:   0%|          | 0.00/233k [00:00<?, ?B/s]

emotion/test-00000-of-00001.parquet:   0%|          | 0.00/105k [00:00<?, ?B/s]

emotion/validation-00000-of-00001.parque(…):   0%|          | 0.00/28.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3257 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1421 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/374 [00:00<?, ? examples/s]

Test set size: 1421


In [ ]:
evaluator_qwen = LLMEvaluator(hf_model_name="Qwen/Qwen3-4B-Instruct-2507")
evaluator_qwen.load_hf_model()

print("Running Qwen with Minimal Prompt...")
res_qwen_p1 = evaluator_qwen.evaluate(test_subset, "hf", PROMPT_1_MINIMAL, num_samples=None)

print("Running Qwen with Structured Prompt...")
res_qwen_p2 = evaluator_qwen.evaluate(test_subset, "hf", PROMPT_2_STRUCTURED, num_samples=None)

Loading HF model: Qwen/Qwen3.5-0.8B on cpu...


config.json: 0.00B [00:00, ?B/s]

You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

ValueError: The checkpoint you are trying to load has model type `qwen3_5` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [ ]:
evaluator_openai = LLMEvaluator(openai_api_key=openai_key)

print("Running GPT-4o-mini with Minimal Prompt...")
res_gpt_p1 = evaluator_openai.evaluate(test_subset, "openai", PROMPT_1_MINIMAL, num_samples=None)

print("Running GPT-4o-mini with Structured Prompt...")
res_gpt_p2 = evaluator_openai.evaluate(test_subset, "openai", PROMPT_2_STRUCTURED, num_samples=None)

In [ ]:
results = {
    "GPT-4o-mini (Minimal)": res_gpt_p1,
    "GPT-4o-mini (Structured)": res_gpt_p2,
    "Qwen-0.8B (Minimal)": res_qwen_p1,
    "Qwen-0.8B (Structured)": res_qwen_p2
}

comparison_df = pd.DataFrame({
    k: {m: v[m] for m in ["Accuracy", "Macro-F1", "Time_per_100"]}
    for k, v in results.items()
}).transpose()

comparison_df.style.highlight_max(subset=["Accuracy", "Macro-F1"], color='lightgreen')

In [ ]:
from datasets import Dataset
import huggingface_hub

hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(token=hf_token)

predictions_data = {
    'True_Labels': results["GPT-4o-mini (Minimal)"]["True_Labels"]
}
for k, v in results.items():
    predictions_data[k] = v["Predictions"]

predictions_dataset = Dataset.from_dict(predictions_data)
predictions_dataset.push_to_hub('bdanko/llm-evaluation-predictions')